In [ ]:
import joblib
import re

#Загрузка моделей
model_views = joblib.load("models/model_views.pkl")
model_likes = joblib.load("models/model_likes.pkl")
feature_columns = joblib.load("models/features.pkl")
category_keywords = joblib.load("models/keywords.pkl")

#Классификация постов
def classify_post(text):
    text = text.lower()
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if kw in text:
                return category
    return "Другое"

#Получение признаков
def extract_features(text, day_of_week, hour):
    category = classify_post(text)
    text_len = len(text)
    emoji_count = len(re.findall(r"[^\w\s,]", text))

    features = {
        "Комментарии": 0,
        "Час": hour,
        "День недели": day_of_week,
        "Длина текста": text_len,
        "Количество эмодзи": emoji_count,
    }

    for col in feature_columns:
        if col.startswith("Категория_"):
            features[col] = 1 if col == f"Категория_{category}" else 0

    for col in feature_columns:
        if col not in features:
            features[col] = 0

    return [features[col] for col in feature_columns], category, text_len, emoji_count

#Ввод
def main():
    print("Введите текст поста:")
    text = input("> ").strip()

    print("Введите день недели (0 — Пн, 6 — Вс):")
    day_of_week = int(input("> ").strip())

    print("Введите час публикации (0–23):")
    hour = int(input("> ").strip())

    features, category, text_len, emoji_count = extract_features(text, day_of_week, hour)

    pred_views = model_views.predict([features])[0]
    pred_likes = model_likes.predict([features])[0]

    print("\n📝 Введённые данные:")
    print(f"  ▶ День недели: {day_of_week} (0=Пн, 6=Вс)")
    print(f"  ▶ Час публикации: {hour}")
    print(f"  ▶ Категория текста: {category}")
    print(f"  ▶ Длина текста: {text_len} символов")
    print(f"  ▶ Кол-во эмодзи: {emoji_count}")

    print("\n📊 Прогноз модели:")
    print(f"  ▶ Ожидаемые просмотры: {int(pred_views):,}")
    print(f"  ▶ Ожидаемые лайки:     {int(pred_likes):,}")

if __name__ == "__main__":
    main()
